# Advanced Retrieval Techniques for RAG

This notebook demonstrates various advanced retrieval techniques used in Retrieval-Augmented Generation (RAG) systems. We will explore different types of retrievers, query transformation methods, and reranking strategies to enhance the relevance and quality of retrieved documents.

### Learning: Install Dependencies

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** First, we need to install the necessary libraries for this notebook, including `langchain_community`, `langchain_text_splitters`, `langchain_openai`, `langchain_chroma`, and `pypdf`.

**Watch for:** Run once; restart runtime if Colab asks.



In [11]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 9.6 MB/s eta 0:00:00


### Learning: Import Libraries

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Next, we import all the required modules from the installed libraries. These include tools for document loading, text splitting, embeddings, vector stores, and various retrieval components.

**Watch for:** If an import fails, re-run the install cell.



In [5]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/tmp/ipykernel_402/2860160806.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: Set Up OpenAI API Key

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** To use OpenAI models for embeddings and chat, we need to set up the API key. We will retrieve it from Google Colab's user data secrets for security.

**Watch for:** If an import fails, re-run the install cell.



In [7]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: if not os.environ.get("OPENAI_API_KEY"):

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** This cell ensures that the OpenAI API key is properly configured. If not found in `userdata`, it will prompt the user to enter it.

**Watch for:** Never hardcode secrets in shared notebooks.



In [8]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: Load Research Paper Data

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** We will load a PDF research paper, specifically the Llama 2 paper, for our retrieval demonstrations. This section handles finding the PDF file and defining its path.

**Watch for:** Confirm page count and first-page text look sane.



In [9]:
DATA_DIR = Path(
    r"/content/data"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
/content/data/llama2-research-paper.pdf


### Learning: loader = PyPDFLoader(str(PDF_PATH))

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Using `PyPDFLoader`, we load the PDF document. Each page of the PDF is treated as a `Document` object.

**Watch for:** Confirm page count and first-page text look sane.



In [12]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 77


### Learning: print("First-page metadata:")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Let's inspect the first page's metadata and content to understand the structure of the loaded documents.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [13]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh K

### Learning: Enrich Document Metadata

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** To facilitate more granular filtering and retrieval, we define a helper function to identify the major section of the paper based on its page number. This allows us to add custom metadata to each document.

**Watch for:** Confirm page count and first-page text look sane.



In [14]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

### Learning: PyPDFLoader page index is normally zero-based

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** We iterate through all loaded pages and update their metadata with additional context like the paper name, organization, year, document type, and the identified section.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

### Learning: for page_document in pages[:5]:

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Let's check the updated metadata for the first few pages to confirm the enrichment.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [16]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/llama2-research-paper.pdf', 'total

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}


### Learning: Chunking Documents

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Large documents need to be split into smaller, manageable chunks for effective retrieval. We use `RecursiveCharacterTextSplitter` to create chunks with specified size and overlap.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


### Learning: for chunk_number, chunk in enumerate(chunks):

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** We assign a unique `chunk_id` to each chunk, incorporating its page number and its order within the page, which can be useful for debugging and tracing.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [18]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

### Learning: print("Chunk content:")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Let's examine the content and metadata of the first chunk to ensure it's structured as expected.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [19]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

### Learning: Generate Embeddings

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Embeddings convert text into numerical vectors, which are essential for semantic search. We use OpenAI's `text-embedding-3-small` model.

**Watch for:** Never hardcode secrets in shared notebooks.



In [20]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### Learning: test_vector = embeddings.embed_query(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** We'll test the embedding model by embedding a sample query and inspecting the dimensions and a few values of the resulting vector.

**Watch for:** Note dimension size; it must match the index.



In [21]:
test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 1536
First 10 values: [0.0028285980224609375, -0.052093505859375, -0.0210418701171875, -0.05535888671875, -0.0264129638671875, 0.028961181640625, -0.0020694732666015625, 0.03472900390625, -0.0164794921875, -0.02447509765625]


### Learning: Create and Persist Vector Store

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** We use Chroma as our vector store to store the document chunks and their embeddings. This allows for efficient similarity search. We also include an option to rebuild the index if needed.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [22]:
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

### Learning: vector_store = Chroma.from_documents(

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** This cell creates the Chroma vector store from our `chunks` and `embeddings`. It will persist the index to disk, so it can be reloaded later without re-embedding.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [23]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 343
Persisted at: /content/data/chroma_llama2_retriever


### Learning: Load Existing Vector Store

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** If the vector store has already been created and persisted, we can load it directly from the disk. This avoids re-processing the embeddings every time the notebook is run.

**Watch for:** Note dimension size; it must match the index.



In [24]:
# Same directory used during creation
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever"

### Learning: PERSIST_DIRECTORY

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Verify the persistence directory path.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [25]:
PERSIST_DIRECTORY

PosixPath('/content/data/chroma_llama2_retriever')

### Learning: Load the existing Chroma collection

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** This cell loads the existing Chroma vector store, ensuring we can access the embedded documents.

**Watch for:** Note dimension size; it must match the index.



In [26]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: /content/data/chroma_llama2_retriever


### Learning: Helper Function to Display Documents

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** We define a utility function `display_documents` to neatly print the retrieved documents, showing their rank, metadata, and a truncated version of their content. This will help in visualizing retrieval results.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [27]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

### Learning: vector_store.similarity_search()

**What you'll learn:** Set knobs (k, window size, weights) before the experiment.

**What this cell does:** The `vector_store.similarity_search()` method can be used for basic retrieval. We will use it within more advanced retriever configurations.

**Watch for:** Change one knob at a time when you compare runs.



In [ ]:
# vector_store.similarity_search()

### Learning: Similarity Search (Dense Retrieval)

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** We configure a `similarity_retriever` using `vector_store.as_retriever()` with `search_type="similarity"`. This type of retriever fetches documents most similar to the query based on their embedding vectors.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [28]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

### Learning: query = "What model sizes of Llama 2 were released?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Let's perform a similarity search with a specific query and display the top 4 results.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [29]:
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-12
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the open release of LLMs, when done safely, will be a net benefit to society. Like all LLMs,
Llama 2 is

RANK: 2
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-342
SOURCE

### Learning: Maximum Marginal Relevance (MMR) Retrieval

**What you'll learn:** Balance relevance with diversity to reduce near-duplicates.

**What this cell does:** MMR search aims to retrieve documents that are both relevant to the query and diverse among themselves. It balances relevance with diversity to avoid redundancy in results.

**Watch for:** Watch lambda_mult: too high ≈ plain similarity; too low ≈ off-topic.



In [30]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

### Learning: query = "How was Llama 2-Chat trained and aligned?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We'll test the MMR retriever with a query, asking how Llama 2-Chat was trained and aligned. This should return relevant yet diverse documents.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [31]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 5
SECTION: pretraining
CHUNK ID: llama2-page-5-chunk-14
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdes

RANK: 2
PAPER PAGE: 31
SECTION: safety
CHUNK ID: llama2-page-31-chunk-135
SOURCE: /

### Learning: query = "How was Llama 2-Chat trained and aligned?"

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Let's compare the results from similarity search and MMR for the same query to observe the diversity introduced by MMR.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [32]:
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

### Learning: print("Similarity Search results:")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This cell prints the paper page, section, and chunk ID for documents returned by both similarity search and MMR, allowing for a direct comparison of their retrieval characteristics.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [33]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
5 pretraining llama2-page-5-chunk-14
3 introduction llama2-page-3-chunk-9
8 fine_tuning llama2-page-8-chunk-29
4 introduction llama2-page-4-chunk-12

MMR results:
5 pretraining llama2-page-5-chunk-14
31 safety llama2-page-31-chunk-135
77 appendix llama2-page-77-chunk-339
34 discussion llama2-page-34-chunk-146


### Learning: Similarity Score Threshold Retrieval

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** This retrieval method only returns documents whose similarity score to the query exceeds a specified threshold. This helps in filtering out less relevant documents.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [34]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.64,
    }
)

### Learning: query = "What safety techniques were used for Llama 2-Chat?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We'll query about safety techniques and see which documents meet the `0.64` score threshold.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [35]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual raters. Additionally, these
safety evaluations are performed using content standards that are likely to be biased towards theLlama
2-Chatmodels.
We are releasing the following models to the general publ



### Learning: threshold_retriever = vector_store.as_retriever(

**What you'll learn:** Return only candidates above a relevance floor.

**What this cell does:** This commented out code block shows an example of how you might adjust the `score_threshold` for different results.

**Watch for:** If results are empty, the threshold is too strict for this query.



In [36]:
# threshold_retriever = vector_store.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 10,
#         "score_threshold": 0.35,
#     }


### Learning: query = "What safety techniques were used for Llama 2-Chat?"

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Instead of using `as_retriever`, we can directly use `similarity_search_with_relevance_scores` on the `vector_store` to get the scores alongside the documents. This provides more control for custom filtering.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [37]:
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.6412
Paper page: 4
Section: introduction
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual ra
Rank: 2
Relevance score: 0.6368
Paper page: 3
Section: introduction
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3
Rank: 3
Relevance score: 0.6297
Paper page: 2
Section: front_matter
4 Safety 20
4.1 Safe

### Learning: Calculating Embeddings Metrics (Cosine Similarity, Euclidean Distance, Dot Product)

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** This section demonstrates how to manually calculate different similarity metrics between a query embedding and document embeddings, which are the underlying mechanisms for vector search.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [38]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


### Learning: query_vector = np.asarray(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Here, we embed the metric query and the candidate document texts to get their numerical vector representations. We then print their shapes to verify.

**Watch for:** Note dimension size; it must match the index.



In [39]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (1536,)
Document-vectors shape: (6, 1536)


### Learning: def cosine_similarity(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** We define functions to calculate Cosine Similarity, Euclidean Distance, and Dot Product between two vectors. These are common metrics used to quantify the similarity between embeddings.

**Watch for:** Note dimension size; it must match the index.



In [40]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

### Learning: metric_rows = []

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This cell iterates through the candidate documents and calculates all three similarity metrics (cosine similarity, Euclidean distance, and dot product) between the query vector and each document vector. The results are stored in a DataFrame for easy comparison.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [41]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,9,fine_tuning,llama2-page-9-chunk-34,0.579142,0.917512,0.579219,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,llama2-page-32-chunk-138,0.538664,0.960562,0.538669,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,llama2-page-17-chunk-73,0.527883,0.971572,0.527725,system message during the conversation by inte...
3,13,fine_tuning,llama2-page-13-chunk-54,0.505156,0.995052,0.505381,evaluating a generative model is an open resea...
4,10,fine_tuning,llama2-page-10-chunk-35,0.500087,0.999762,0.499936,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,llama2-page-10-chunk-40,0.495792,1.004133,0.495728,3.2.2 Reward Modeling The reward model takes a...


### Learning: metric_query

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Display the metric query for reference.

**Watch for:** Use the metric your embedding model was trained with.



In [42]:
metric_query

'How was reinforcement learning with human feedback used?'

### Learning: metric_table.sort_values(

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Sort the results by cosine similarity in descending order to see the most similar documents first. Cosine similarity ranges from -1 (opposite) to 1 (identical).

**Watch for:** Use the metric your embedding model was trained with.



In [43]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,9,fine_tuning,0.579142,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538664,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527883,system message during the conversation by inte...
3,13,fine_tuning,0.505156,evaluating a generative model is an open resea...
4,10,fine_tuning,0.500087,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495792,3.2.2 Reward Modeling The reward model takes a...


### Learning: metric_table.sort_values(

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Sort the results by Euclidean distance in ascending order. Smaller Euclidean distance indicates higher similarity.

**Watch for:** Use the metric your embedding model was trained with.



In [44]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,9,fine_tuning,0.917512,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.960562,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.971572,system message during the conversation by inte...
3,13,fine_tuning,0.995052,evaluating a generative model is an open resea...
4,10,fine_tuning,0.999762,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,1.004133,3.2.2 Reward Modeling The reward model takes a...


### Learning: metric_table.sort_values(

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Sort the results by dot product in descending order. For normalized vectors, dot product is equivalent to cosine similarity.

**Watch for:** Use the metric your embedding model was trained with.



In [45]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,9,fine_tuning,0.579219,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538669,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527725,system message during the conversation by inte...
3,13,fine_tuning,0.505381,evaluating a generative model is an open resea...
4,10,fine_tuning,0.499936,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495728,3.2.2 Reward Modeling The reward model takes a...


### Learning: normalized_query_vector = (

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** This section explicitly demonstrates that when vectors are normalized, the dot product between them becomes equivalent to their cosine similarity. This is a fundamental concept in vector space models.

**Watch for:** Use the metric your embedding model was trained with.



In [46]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.57914198 0.53866448 0.52788284 0.50515638 0.50008706 0.49579245]

Dot product after normalization:
[0.57914198 0.53866448 0.52788284 0.50515638 0.50008706 0.49579245]

Are they approximately equal? True


### Learning: Filtered Retrieval (Pre-filtering)

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** We can apply filters *before* the similarity search to restrict the search space to documents matching specific metadata criteria (e.g., only documents from the 'fine_tuning' section). This is known as pre-filtering.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [47]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

### Learning: query = "How was Llama 2-Chat aligned with human preferences?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Let's query how Llama 2-Chat was aligned with human preferences, but only retrieve documents from the `fine_tuning` section.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [48]:
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 19
SECTION: fine_tuning
CHUNK ID: llama2-page-19-chunk-79
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure12: Humanevaluationresults for Llama 2-Chatmodelscomparedtoopen-andclosed-sourcemodels
across ~4,000 helpfulness prompts with three raters per prompt.
The largestLlama 2-Chat model is competitive with ChatGPT.Llama 2-Chat 70B model has a win rate of
36% and a tie rate of 31.5% relative to ChatGPT.Llama 2-Chat 70B model outperforms PaLM-bison chat
model by a large percentage on our prompt set. More results and analysis is available in Section A.3.7.
Inter-Rater Reliability (IRR). In our human evaluations, three different annotators provided independent
assessments for each model generation comparison. High IRR scores (closer to 1.0) are typically seen as
better from a data quality persp

RANK: 2
PAPER PAGE: 18
SECTION: fine_tuning
CHUNK ID: llama2-page-18-chunk-77
SOU

### Learning: for document in fine_tuning_documents:

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This assertion verifies that all retrieved documents indeed belong to the 'fine_tuning' section, confirming the filter's effectiveness.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [49]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


### Learning: filtered_retriever = vector_store.as_retriever(

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Here's a more complex pre-filter example using multiple conditions (`$and`): filtering by `section`, `year`, and `organization`.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [50]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

### Learning: query = "How was human preference data collected?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We query about human preference data collection, applying the complex pre-filter to retrieve only relevant documents.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [51]:
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two 

RANK: 2
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-38
SOU

Prefilter

## 16. Filtered Retrieval (Post-filtering)

Post-filtering involves retrieving a larger set of candidate documents first, and then applying metadata filters *after* the initial retrieval. This can be useful when the filter conditions are complex or when the vector store doesn't support advanced pre-filtering efficiently.

### Learning: pre_filter = {

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** This cell sets up a pre-filter using the same criteria as before and demonstrates a retriever configured with this pre-filter.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [52]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK: 2
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-42
SOU

Post-Filtering

## 17. Post-filtering Example

First, we retrieve a broader set of candidates without any filters directly applied to the retriever. This retrieves documents from various sections.

### Learning: unfiltered_candidates = vector_store.similarity_search(

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** We fetch 15 candidate documents without any initial filtering.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [53]:
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

### Learning: post_filtered_documents = [

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** After retrieving the broad set of documents, we manually apply the filtering conditions (section and year) in Python. This is post-filtering.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [54]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK: 2
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-42
SOU

### Learning: print(

**What you'll learn:** Score broadly, then drop non-matching candidates.

**What this cell does:** This cell compares the number of documents before and after post-filtering, illustrating how post-filtering reduces the set of candidate documents to only those meeting the specified criteria.

**Watch for:** Post-filter can leak restricted docs into the candidate set.



In [55]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 13


05-Aug-2026

## 18. Hybrid Retrieval: Combining Sparse and Dense Search

This section introduces `EnsembleRetriever`, which combines the strengths of sparse (e.g., BM25) and dense (e.g., vector search) retrieval methods. Sparse retrieval excels at keyword matching, while dense retrieval captures semantic meaning.

### Learning: import os

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** We import additional components for advanced retrieval, including `BM25Retriever`, `EnsembleRetriever`, `ChatOpenAI` for query transformations, and `ContextualCompressionRetriever` with `CrossEncoderReranker`.

**Watch for:** If an import fails, re-run the install cell.



In [56]:
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

### Learning: def display_documents(

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This is an updated `display_documents` function that includes a title and limits the number of documents displayed, making it more flexible for showcasing different retrieval results.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

### Learning: def deduplicate_documents(documents):

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** A helper function to remove duplicate documents while preserving the order. This is particularly useful when combining results from multiple retrievers.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

### Learning: from langchain_community.retrievers import BM25Retriever

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Import `BM25Retriever` for sparse search.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from langchain_community.retrievers import BM25Retriever

### Learning: 1. BM25 Sparse Retrieval

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** We initialize a `BM25Retriever` from our `chunks`. BM25 is a ranking function used in information retrieval that estimates the relevance of documents to a given search query.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)

### Learning: bm25_retriever

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Inspect the BM25 retriever object.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000216948DD9D0>)

### Learning: Final number of results

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Set the number of top documents `k` that the BM25 retriever should return.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# Final number of results
bm25_retriever.k = 4

### Learning: sparse_query = "Grouped-Query Attention GQA 70B"

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Define a sparse query that includes keywords to test the BM25 retriever.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
sparse_query = "Grouped-Query Attention GQA 70B"

### Learning: sparse_documents = bm25_retriever.invoke(sparse_query)

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Invoke the BM25 retriever with the sparse query to get the relevant documents.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
sparse_documents = bm25_retriever.invoke(sparse_query)

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the documents retrieved by the BM25 sparse retriever.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 15

RANK: 2
Paper page: 48
Section: appendix
Chunk ID: llama2-page-48-chunk-220
----------------------------------------------------------------------------------------------------
BoolQ PIQA 

### Learning: 2. Dense Vector Retrieval

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** We configure a `dense_retriever` using the `vector_store.as_retriever()` with `search_type="similarity"`. This leverages the semantic understanding from embeddings.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

### Learning: dense_query = (

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Define a dense query to test the vector-based retriever and invoke it.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-15
----------------------------------------------------------------------------------------------------
Specifically, we performed more robust data cleaning, updated our data mixes, trained on 40% more total
tokens,doubledthecontextlength,andusedgrouped-queryattention(GQA)toimproveinferencescalability
for our larger models. Table 1 compares the attributes of the newLlama 2 models with theLlama 1 models.
2.1 Pretraining Data
Our training corpus includes a new mix of data from publicly available sources, which does not include data
from Meta’s products or services. We made an effort to remove data from certain sites known to contain a
high volume of personal information about private individuals. 

RANK: 2
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-340
----------------------------------------------------------------------------------------------------
Lla

### Learning: 3. Comparing Sparse and Dense Retrieval

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Let's compare the results of BM25 (sparse) and dense (vector) retrieval for the same query. This highlights their different strengths in capturing keyword matching versus semantic similarity.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow 

### Learning: print("SPARSE RESULTS")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This cell prints the rank, paper page, and chunk ID for both sparse and dense retrieval results, allowing for a clear side-by-side comparison.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 4 llama2-page-4-chunk-12
2 47 llama2-page-47-chunk-217
3 54 llama2-page-54-chunk-240
4 6 llama2-page-6-chunk-18

DENSE RESULTS
1 13 llama2-page-13-chunk-53
2 48 llama2-page-48-chunk-221
3 5 llama2-page-5-chunk-15
4 6 llama2-page-6-chunk-18


### Learning: bm25_retriever.k = 8

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** To prepare for hybrid retrieval, we adjust the `k` parameter for both BM25 and dense retrievers to fetch more candidates, which will then be combined.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

### Learning: 4. Ensemble Retriever (Hybrid Retrieval with RRF)

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** An `EnsembleRetriever` combines multiple retrievers, such as BM25 and dense search, using techniques like Reciprocal Rank Fusion (RRF) to create a single, more robust set of results. This leverages the strengths of both keyword-based and semantic-based search.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

### Learning: hybrid_query = (

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Define a query for hybrid retrieval, focusing on a topic that benefits from both keyword and semantic matching.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
hybrid_query = (
    "Llama 2 70B grouped-query attention and inference scalability"
)

### Learning: hybrid_documents = hybrid_retriever.invoke(hybrid_query)

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Invoke the `hybrid_retriever` with the defined query to get the combined results.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the documents returned by the hybrid retriever, showcasing the combined ranking from BM25 and dense search using RRF.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Trai

                         ┌── BM25 Retriever ─────┐
User query ──────────────┤                       ├── Weighted RRF
                         └── Dense Retriever ────┘
                                                    ↓
                                            Combined ranking

This diagram illustrates the process of hybrid retrieval using BM25 and Dense Retriever combined with Weighted Reciprocal Rank Fusion (RRF).

### Learning: Query Rewriting

**What you'll learn:** Turn chatty follow-ups into a standalone search query.

**What this cell does:** Query rewriting is a technique where a conversational or ambiguous query is transformed into a clear, standalone search query. This is especially useful in conversational AI to maintain context over multiple turns.

**Watch for:** Log original + rewrite; rewriters can hallucinate entities.



In [ ]:
test_query = (
    "How did Meta make Llama 2 70B efficient for large-scale inference?"
)

### Learning: sparse_results = bm25_retriever.invoke(test_query)

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** This cell performs sparse, dense, and hybrid retrieval for a `test_query` to set up a comparison for query transformation techniques.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the results from sparse, dense, and hybrid retrievers for the `test_query`. This provides a baseline before applying query rewriting.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 54
Section: appendix
Chunk ID: llama2-page-54-chunk-240
----------------------------------------------------------------------------------------------------
attribute, and so, up to 20 turns (we did not extend the human evaluation more, and all the examples had
less than 4048 tokens in total over the turns). As a comparison,Llama 2-Chat without GAtt can not anymore
refer to the attributes after only few turns: from 100% at turn t+1, to 10% at turn t+3 and then 0%.
GAtt Zero-shot Generalisation. We tried at inference time to set constrain not present in the training of
GAtt. For instance, “answer in one sentence only”, for which the model remained consistent, as illustrated in
Figure 28.
We applied first GAtt toLlama 1, which was pretrained with a 

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params C

### Learning: CHAT_MODEL = os.environ.get(

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** We initialize a `ChatOpenAI` model to be used for query rewriting. It's configured to be deterministic (`temperature=0`).

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

Chat model: gpt-4.1-mini


### Learning: query_rewriting_prompt = ChatPromptTemplate.from_messages(

**What you'll learn:** Turn chatty follow-ups into a standalone search query.

**What this cell does:** This `ChatPromptTemplate` defines the instructions for the LLM to rewrite a conversational query into a standalone search query, preserving important entities and resolving pronouns.

**Watch for:** Log original + rewrite; rewriters can hallucinate entities.



In [ ]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

### Learning: query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser

**What you'll learn:** Turn chatty follow-ups into a standalone search query.

**What this cell does:** We create a LangChain chain for query rewriting, combining the prompt, LLM, and a string output parser.

**Watch for:** Log original + rewrite; rewriters can hallucinate entities.



In [ ]:
query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser())

### Learning: chat_history = """

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Define a sample chat history to provide context for the conversational query.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
chat_history = """
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
"""

### Learning: original_query = "What did Meta do after that?"

**What you'll learn:** Turn chatty follow-ups into a standalone search query.

**What this cell does:** Define the original conversational query that needs to be rewritten.

**Watch for:** Log original + rewrite; rewriters can hallucinate entities.



In [ ]:
original_query = "What did Meta do after that?"

### Learning: rewritten_query = query_rewriting_chain.invoke(

**What you'll learn:** Turn chatty follow-ups into a standalone search query.

**What this cell does:** Invoke the `query_rewriting_chain` with the chat history and original query to get the rewritten query.

**Watch for:** Log original + rewrite; rewriters can hallucinate entities.



In [ ]:
rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

### Learning: print("Original query:")

**What you'll learn:** Turn chatty follow-ups into a standalone search query.

**What this cell does:** Print the original and rewritten queries to demonstrate the transformation.

**Watch for:** Log original + rewrite; rewriters can hallucinate entities.



In [ ]:
print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

Original query:
What did Meta do after that?

Rewritten query:
What steps did Meta take after the initial supervised fine-tuning of Llama 2-Chat?


### Learning: rewritten_query_documents = hybrid_retriever.invoke(

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Now, we use the `hybrid_retriever` with the `rewritten_query` to see if the reformulated query yields better or more relevant results.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the documents retrieved using the rewritten query. Compare these results with those from the original query if available.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)


Documents Retrieved Using the Rewritten Query

RANK: 1
Paper page: 8
Section: fine_tuning
Chunk ID: llama2-page-8-chunk-29
----------------------------------------------------------------------------------------------------
are from OpenAI (2023). Results for the PaLM model are from Chowdhery et al. (2022). Results for the
PaLM-2-L are from Anil et al. (2023).
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.
In this section, we report on our experiments and findings using supervised fine-tuning (Section 3.1), as
well as initial and iterative reward modeling (Section 3.2.2) and RLHF (Section 3.2.3). We also share a
new technique, Ghost A

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
-------------------------------------------------------------------------------------------------

### Learning: Query Expansion

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Query expansion involves generating multiple alternative versions of a user's query to broaden the search and retrieve more comprehensive results. This increases the chances of matching relevant documents that might use different terminology.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

### Learning: query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** We configure the LLM to output a structured Pydantic object `ExpandedQueryOutput`, ensuring the output is a list of alternative queries.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

### Learning: query_expansion_prompt = ChatPromptTemplate.from_messages(

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** This `ChatPromptTemplate` instructs the LLM to generate four alternative search queries using synonyms, technical terms, and alternative wording, without answering the query itself.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

### Learning: query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Create a LangChain chain for query expansion by combining the prompt and the structured output LLM.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

### Learning: original_query = (

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Define the original query for which we want to generate expanded versions.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)


### Learning: expanded_output = query_expansion_chain.invoke(

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Invoke the `query_expansion_chain` to get the `ExpandedQueryOutput` object containing the alternative queries.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

### Learning: expanded_output

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Display the raw output of the `expanded_output` object.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
expanded_output

ExpandedQueryOutput(queries=['How was Llama 2-Chat enhanced through human feedback?', 'In what ways did human feedback improve Llama 2-Chat?', 'What improvements were made to Llama 2-Chat using human-in-the-loop feedback?', 'How did human feedback contribute to the development of Llama 2-Chat?'])

### Learning: all_expanded_queries = [

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Combine the original query with all the generated expanded queries into a single list.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]


### Learning: all_expanded_queries

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Display the complete list of queries, including the original and its expanded versions.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
all_expanded_queries

['How was Llama 2-Chat improved using human feedback?',
 'How was Llama 2-Chat enhanced through human feedback?',
 'In what ways did human feedback improve Llama 2-Chat?',
 'What improvements were made to Llama 2-Chat using human-in-the-loop feedback?',
 'How did human feedback contribute to the development of Llama 2-Chat?']

### Learning: print("Generated search queries:\n")

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Print the generated search queries in a readable format.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

Generated search queries:

1. How was Llama 2-Chat improved using human feedback?
2. How was Llama 2-Chat enhanced through human feedback?
3. In what ways did human feedback improve Llama 2-Chat?
4. What improvements were made to Llama 2-Chat using human-in-the-loop feedback?
5. How did human feedback contribute to the development of Llama 2-Chat?


### Learning: expanded_query_documents = []

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We iterate through all expanded queries, perform a dense retrieval for each, combine the results, and then deduplicate them to get a comprehensive set of unique relevant documents.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)


Query Expansion: Combined Unique Documents

RANK: 1
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within d

RANK: 2
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
---------------------------------------------------------------------------------------------------

### Learning: 1. MultiQueryRetriever (Built-in Query Expansion)

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** LangChain provides a built-in `MultiQueryRetriever` that automates the query expansion process using an LLM. It generates multiple queries from a single input query and retrieves documents for all of them.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

### Learning: multi_query_retriever = MultiQueryRetriever.from_llm(

**What you'll learn:** Ask the corpus several phrasings and pool the hits.

**What this cell does:** Initialize the `MultiQueryRetriever`, passing our `dense_retriever` and `llm`. We set `include_original=True` to also use the original query.

**Watch for:** Compare against a single dense query on the same question.



In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

### Learning: multi_query_documents = multi_query_retriever.invoke(

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Invoke the `multi_query_retriever` with a query to demonstrate its functionality.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
multi_query_documents = multi_query_retriever.invoke(
    "How did human feedback improve Llama 2-Chat?"
)


### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the documents retrieved by the `MultiQueryRetriever`. This combines results from multiple generated queries.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)


Built-in MultiQueryRetriever Results

RANK: 1
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedba

### Learning: Query Decomposition

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** Query decomposition breaks down a complex, multi-part query into several simpler, independent sub-queries. Each sub-query can then be used to retrieve specific pieces of information, and the results are combined to answer the original complex query.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

### Learning: query_decomposition_llm = llm.with_structured_output(

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** We configure the LLM to output a structured Pydantic object `DecomposedQueryOutput`, ensuring the output is a list of sub-queries.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

### Learning: query_decomposition_prompt = ChatPromptTemplate.from_messages(

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** This `ChatPromptTemplate` instructs the LLM to decompose a complex query into independent, atomic search queries, preserving named entities and model names.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

### Learning: query_decomposition_chain = (

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** Create a LangChain chain for query decomposition by combining the prompt and the structured output LLM.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
query_decomposition_chain = (
    query_decomposition_prompt
    | query_decomposition_llm
)

### Learning: complex_query = """

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** Define a complex query that involves multiple aspects, making it suitable for decomposition.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
complex_query = """
Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.
"""

### Learning: decomposed_output = query_decomposition_chain.invoke(

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** Invoke the `query_decomposition_chain` to get the `DecomposedQueryOutput` object containing the sub-queries.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)


### Learning: sub_queries = decomposed_output.sub_queries

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** Extract the list of sub-queries from the `decomposed_output`.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
sub_queries = decomposed_output.sub_queries

### Learning: print("Original complex query:")

**What you'll learn:** Split a complex question into atomic sub-queries.

**What this cell does:** Print the original complex query and the generated sub-queries to illustrate the decomposition.

**Watch for:** Missing a sub-query silently produces an incomplete answer.



In [ ]:
print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

Original complex query:

Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.


Generated sub-queries:
1. What is the pretraining process of Llama 2?
2. What is the fine-tuning process of Llama 2-Chat?
3. How does Meta improve model safety in Llama 2 and Llama 2-Chat?


### Learning: decomposition_results = {}

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** For each generated sub-query, perform a hybrid retrieval using the `hybrid_retriever` and store the results.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

### Learning: for sub_query, documents in decomposition_results.items():

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the retrieved documents for each sub-query. This shows how different parts of the complex query are addressed by individual searches.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )


Sub-query: What is the pretraining process of Llama 2?

RANK: 1
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-215
----------------------------------------------------------------------------------------------------
• Armand Joulin, Edouard Grave, Guillaume Lample, and Timothee Lacroix, members of the original
Llama team who helped get this work started.
• Drew Hamlin, Chantal Mora, and Aran Mun, who gave us some design input on the figures in the
paper.
• Vijai Mohan for the discussions about RLHF that inspired our Figure 20, and his contribution to the
internal demo.
• Earlyreviewersofthispaper,whohelpedusimproveitsquality,includingMikeLewis,JoellePineau,
Laurens van der Maaten, Jason Weston, and Omer Levy.
A.2 Additional Details for Pretraining
A.2.1 Architecture Changes Compared toLlama 1
Context Leng

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
----------------------------------------------------------------------------------------

### Learning: all_decomposition_documents = []

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Combine all documents retrieved from the sub-queries into a single list and remove duplicates to get a comprehensive set of evidence.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)


Combined Evidence from All Sub-Queries

RANK: 1
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-215
----------------------------------------------------------------------------------------------------
• Armand Joulin, Edouard Grave, Guillaume Lample, and Timothee Lacroix, members of the original
Llama team who helped get this work started.
• Drew Hamlin, Chantal Mora, and Aran Mun, who gave us some design input on the figures in the
paper.
• Vijai Mohan for the discussions about RLHF that inspired our Figure 20, and his contribution to the
internal demo.
• Earlyreviewersofthispaper,whohelpedusimproveitsquality,includingMikeLewis,JoellePineau,
Laurens van der Maaten, Jason Weston, and Omer Levy.
A.2 Additional Details for Pretraining
A.2.1 Architecture Changes Compared toLlama 1
Context Leng

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
----------------------------------------------------------------------------------------------------
Fig

Complex query
      ↓
Atomic sub-queries
      ↓
Retrieve for every sub-query
      ↓
Merge evidence
      ↓
Remove duplicates

This diagram illustrates the workflow for query decomposition, retrieval for sub-queries, and merging of evidence.

### Learning: Reranking with Cross-Encoder

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Reranking is a post-retrieval step that reorders an initial set of candidate documents to improve their relevance to the query. Cross-encoders are powerful models that can evaluate the relevance of a query-document pair more accurately than traditional methods by considering b...

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

Vector search
     ↓
Top 20 candidates

This diagram illustrates the process of an initial vector search to get top candidates.

### Learning: cross_encoder = HuggingFaceCrossEncoder(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** We initialize a `HuggingFaceCrossEncoder` model, specifying its name and device (`cpu` for demonstration). This cross-encoder will be used to score the relevance of retrieved documents.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sunny\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13097.84it/s]


### Learning: cross_encoder_reranker = CrossEncoderReranker(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** We wrap the cross-encoder model in a `CrossEncoderReranker` and specify `top_n=5` to keep only the top 5 reranked documents.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

### Learning: reranking_retriever = ContextualCompressionRetriever(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** A `ContextualCompressionRetriever` is used to apply the `CrossEncoderReranker` to the results of a `base_retriever` (our `candidate_retriever`).

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

### Learning: reranking_query = (

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** Define a query to test the reranking process.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
reranking_query = (
    "How did Meta collect and use human preference data to train Llama 2-Chat?"
)

### Learning: initial_candidates = candidate_retriever.invoke(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** First, retrieve a larger set of initial candidates using the `candidate_retriever` before reranking. This provides the pool of documents for the cross-encoder to reorder.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

### Learning: initial_candidates

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Display the initial candidates retrieved, noting their original order based on the vector similarity.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
initial_candidates

[Document(id='577892fb-3838-4c4d-9978-4488765bf837', metadata={'producer': 'pdfTeX-1.40.25', 'year': 2023, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creationdate': '2023-07-20T00:30:36+00:00', 'paper_page': 10, 'paper': 'Llama 2', 'title': '', 'page': 9, 'keywords': '', 'section': 'fine_tuning', 'chunk_id': 'llama2-page-10-chunk-38', 'author': '', 'document_type': 'research_paper', 'moddate': '2023-07-20T00:30:36+00:00', 'total_pages': 77, 'subject': '', 'creator': 'LaTeX with hyperref', 'start_index': 2384, 'page_label': '10', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'trapped': '/False', 'organization': 'Meta', 'access_level': 'public'}, page_content='can be found in Section 4.2.1.\nHuman annotations were collected in batches on a weekly basis. As we collected more preference data, our\nreward models improved, and we were abl

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the initial candidates retrieved before reranking. This provides a baseline to compare against the reranked results.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)


Before Reranking: Initial Vector Candidates

RANK: 1
Paper page: 10
Section: fine_tuning
Chunk ID: llama2-page-10-chunk-38
----------------------------------------------------------------------------------------------------
can be found in Section 4.2.1.
Human annotations were collected in batches on a weekly basis. As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distribution, i.e., from
hyper-specialization (Scialom et al., 2020b), it is important before a newLlama 2-Chat tuning iteration to
gather new preference data using the latest

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
-------------------------------------------------------------------------------------------------

### Learning: reranked_documents = reranking_retriever.invoke(reranking_query)

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** Invoke the `reranking_retriever` with the query. This will first fetch candidates using the base retriever and then apply the cross-encoder reranker to select the top `n` most relevant documents.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
reranked_documents = reranking_retriever.invoke(reranking_query)

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the documents after reranking. Observe how their order might have changed compared to the initial candidates, reflecting the cross-encoder's relevance assessment.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)


After Reranking: Final Top Documents

RANK: 1
Paper page: 10
Section: fine_tuning
Chunk ID: llama2-page-10-chunk-38
----------------------------------------------------------------------------------------------------
can be found in Section 4.2.1.
Human annotations were collected in batches on a weekly basis. As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distribution, i.e., from
hyper-specialization (Scialom et al., 2020b), it is important before a newLlama 2-Chat tuning iteration to
gather new preference data using the latest

RANK: 2
Paper page: 11
Section: fine_tuning
Chunk ID: llama2-page-11-chunk-45
----------------------------------------------------------------------------------------------------
t

User query
     ↓
Dense Retriever
     ↓
Top 20 candidates
     ↓
Cross-Encoder Reranker
     ↓
Query-document relevance evaluation
     ↓
Final top 5 documents

This diagram illustrates the complete reranking workflow: initial vector search, candidate selection, cross-encoder evaluation, and final top documents.

### Learning: Hybrid Retrieval + Reranking

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** This combines the benefits of hybrid retrieval (sparse + dense) for broad initial candidate generation with the precision of cross-encoder reranking. This is often the most robust retrieval strategy for complex RAG systems.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
bm25_retriever.k = 15

### Learning: dense_candidate_retriever = vector_store.as_retriever(

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Adjust `k` for the BM25 retriever to fetch more candidates, suitable for an initial broad retrieval.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

### Learning: hybrid_candidate_retriever = EnsembleRetriever(

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Configure a `dense_candidate_retriever` to fetch more candidates, complementing the BM25 retriever.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

### Learning: hybrid_reranking_retriever = ContextualCompressionRetriever(

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Create a `hybrid_candidate_retriever` by combining BM25 and dense retrievers using `EnsembleRetriever` with equal weights. This generates a diverse set of initial candidates.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

### Learning: query = (

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Wrap the `hybrid_candidate_retriever` with a `ContextualCompressionRetriever` and our `cross_encoder_reranker`. This pipeline will first retrieve candidates using hybrid search and then rerank them.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
query = (
    "What techniques did Meta use to improve the helpfulness and safety of Llama 2-Chat?"
)

### Learning: final_documents = hybrid_reranking_retriever.invoke(query)

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Define the final complex query to be used with the hybrid retrieval and reranking pipeline.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
final_documents = hybrid_reranking_retriever.invoke(query)

### Learning: final_documents

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Invoke the `hybrid_reranking_retriever` with the query to get the final, highly relevant documents after both hybrid retrieval and cross-encoder reranking.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
final_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public', 'start_index': 823, 'chunk_id': 'llama2-page-1-chunk-1'}, page_content='Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic\nSergey Edunov Thomas Scialom∗\nGenAI, Meta\nAbstract\nIn t

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the `final_documents` to observe the results of the complete hybrid retrieval and reranking process.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)


Hybrid Retrieval + Cross-Encoder Reranking

RANK: 1
Paper page: 1
Section: front_matter
Chunk ID: llama2-page-1-chunk-1
----------------------------------------------------------------------------------------------------
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey Edunov Thomas Scialom∗
GenAI, Meta
Abstract
In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned
large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters.
Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases. Our
models outperform open-source chat models on most benchmarks we tested, and based on
our human evaluations for helpfulness and 

RANK: 2
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
----------------------------------------------------------------------------------------------------

                         ┌── BM25 Search ───────┐
User query ──────────────┤                      ├── Weighted RRF
                         └── Dense Search ──────┘
                                                   ↓
                                          Candidate documents
                                                   ↓
                                       Cross-Encoder Reranker
                                                   ↓
                                           Final top documents

This diagram illustrates the comprehensive retrieval and reranking pipeline: initial hybrid search, candidate selection, cross-encoder reranking, and final top documents.

Sparse Retrieval
    = BM25Retriever

Dense Retrieval
    = VectorStoreRetriever

Hybrid Retrieval
    = BM25 + Dense + EnsembleRetriever + Weighted RRF

Query Rewriting
    = Convert a contextual query into a standalone query

Query Expansion
    = Generate related query variations and merge their results

Query Decomposition
    = Split one complex query into atomic searchable queries

Reranking
    = Retrieve broad candidates and reorder them using a cross-encoder

## Summary of Retrieval Techniques

This section provides a quick overview of the different retrieval and query transformation techniques explored in this notebook.

1. BM25 sparse retrieval
2. Dense vector retrieval
3. Hybrid retrieval
4. Query rewriting
5. Query expansion
6. Query decomposition
7. Dense retrieval + reranking
8. Hybrid retrieval + reranking

## Retrieval Strategies Covered

Here's a recap of the retrieval strategies and query transformations demonstrated:

HOME_WORK
weighted fusion
resiporcal rank fusion
multi query retriever
Multi hop retriever
HYDE
parenet document retriever
sentence window retriever
contextual compression

CUSTOM_RETRIEVER
Langchain
langchain provide one baseretriever class ontop of it you can create custom retriever with your own logic(when you are usingh langchain/langgraph)

## Homework and Further Exploration

For self-learning and to deepen your understanding, consider exploring the following advanced retrieval techniques and concepts:

### Advanced Retrieval Techniques:
*   **Weighted Fusion:** Custom weighting of different retriever results.
*   **Reciprocal Rank Fusion (RRF):** An algorithm to combine ranked lists from multiple retrieval systems.
*   **Multi-Query Retriever:** Generates multiple distinct queries from a single user input to broaden retrieval.
*   **Multi-Hop Retriever:** Chains multiple retrieval steps, where the result of one query informs the next.
*   **HYDE (Hypothetical Document Embeddings):** Generates a hypothetical answer first, then embeds it to find similar documents.
*   **Parent Document Retriever:** Retrieves larger 'parent' documents after finding relevant small 'child' chunks.
*   **Sentence Window Retriever:** Expands context around retrieved sentences by adding surrounding sentences.
*   **Contextual Compression:** Compresses irrelevant parts of retrieved documents to focus on key information.

### Custom Retriever Implementation:
*   **Custom Retriever with LangChain:** LangChain provides a base `BaseRetriever` class. You can extend this class to create custom retrieval logic tailored to specific needs within LangChain or LangGraph workflows.

### Learning: Next Steps

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** This marks the end of our in-depth session on advanced retrieval techniques. The next step will involve discussing prompting strategies.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
i will take one more hour reteriever
will discuss about the prompting

multimodal RAG builder
full flede project

## Looking Ahead: Building a Multimodal RAG System

In future sessions, we will delve into building a full-fledged multimodal RAG (Retrieval-Augmented Generation) project, integrating various data types and retrieval methods.

MEGA ASSISGNMENT on Sunday

## Major Assignment Coming Soon

Prepare for a comprehensive assignment that will test your understanding and practical application of RAG concepts and techniques discussed in this course.

Agent(evalaution+mcp_guadrails) LLMOPS

## Agent Development and LLM Operations

Beyond RAG, we will also explore agent development, including evaluation metrics and guardrails for LLM operations (LLMOps), ensuring robust and reliable AI systems.